In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
df = pd.read_csv("../data/Crop_recommendation.csv")
X = df[FEATURES]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [2]:
print(df.shape)
print(df.columns.tolist())
print(X_train.shape, X_test.shape)

(2200, 8)
['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
(1760, 7) (440, 7)


In [3]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

nb = GaussianNB()
nb.fit(X_train, y_train)
print("Naive Bayes:", round(accuracy_score(y_test, nb.predict(X_test)), 4))

Naive Bayes: 0.9955


In [4]:
from sklearn.tree import DecisionTreeClassifier

for depth in [2, 4, 6, 10, None]:    # None = no depth limit
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, tree.predict(X_train))
    test_acc = accuracy_score(y_test, tree.predict(X_test))   # unseen rows: the honest score
    print(depth, round(train_acc, 4), round(test_acc, 4))

2 0.1364 0.1364
4 0.3182 0.3182
6 0.583 0.5705
10 0.9818 0.9636
None 1.0 0.9795


In [5]:
from sklearn.metrics import classification_report

pred = nb.predict(X_test)
report = pd.DataFrame(classification_report(y_test, pred, output_dict=True)).T
report = report.drop(["accuracy", "macro avg", "weighted avg"])
print(report.sort_values("f1-score").head(5))

           precision  recall  f1-score  support
rice        1.000000     0.9  0.947368     20.0
jute        0.909091     1.0  0.952381     20.0
blackgram   1.000000     1.0  1.000000     20.0
apple       1.000000     1.0  1.000000     20.0
coconut     1.000000     1.0  1.000000     20.0


In [6]:
from sklearn.metrics import confusion_matrix

labels = nb.classes_
cm = confusion_matrix(y_test, pred, labels=labels)
pairs = [(labels[i], labels[j], cm[i][j])
         for i in range(len(labels)) for j in range(len(labels))
         if i != j and cm[i][j] > 0]
pairs.sort(key=lambda p: p[2], reverse=True)
print(pairs[:5])

[(np.str_('rice'), np.str_('jute'), np.int64(2))]


In [7]:
proba = nb.predict_proba(X_test)
confidence = proba.max(axis=1)
correct = (pred == y_test.values)
print("avg confidence when right:", confidence[correct].mean().round(3))
print("avg confidence when wrong:", confidence[~correct].mean().round(3))

avg confidence when right: 0.993
avg confidence when wrong: 0.788


In [8]:
import numpy as np
rng = np.random.default_rng(42)

def stability(model, X, noise):
    shaken = X + rng.normal(0, noise * X.std().values, X.shape)
    return (model.predict(shaken) == model.predict(X)).mean()

for noise in [0.01, 0.05, 0.1, 0.2]:
    print(noise, round(stability(nb, X_test, noise), 4))

0.01 0.9955
0.05 0.9955
0.1 0.9841
0.2 0.8841


In [9]:
from sklearn.ensemble import IsolationForest

detector = IsolationForest(contamination=0.01, random_state=42).fit(X_train)
weird = pd.DataFrame([[500, 500, 500, 60, 100, 14, 1000]], columns=FEATURES)
print("normal rows flagged:", (detector.predict(X_test) == -1).mean().round(3))
print("weird row flagged:", detector.predict(weird)[0] == -1)

normal rows flagged: 0.014
weird row flagged: True


In [10]:
rng = np.random.default_rng(42)

def stability(model, X, noise):
    shaken = X + rng.normal(0, noise * X.std().values, X.shape)
    return (model.predict(shaken) == model.predict(X)).mean()

for noise in [0.01, 0.05, 0.1, 0.2]:
    print(noise, round(stability(nb, X_test, noise), 4))

0.01 0.9955
0.05 0.9955
0.1 0.9841
0.2 0.8841


In [11]:
for t in [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]:
    flagged = confidence < t
    print(t, "good predictions flagged:", flagged[correct].mean().round(3),
          "| wrong predictions caught:", flagged[~correct].mean().round(3))

0.5 good predictions flagged: 0.0 | wrong predictions caught: 0.0
0.7 good predictions flagged: 0.007 | wrong predictions caught: 0.0
0.8 good predictions flagged: 0.014 | wrong predictions caught: 0.5
0.9 good predictions flagged: 0.016 | wrong predictions caught: 1.0
0.95 good predictions flagged: 0.03 | wrong predictions caught: 1.0
0.99 good predictions flagged: 0.057 | wrong predictions caught: 1.0


In [12]:
dt = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("tree confidence values:", np.unique(dt.predict_proba(X_test).max(axis=1)))
for noise in [0.01, 0.05, 0.1, 0.2]:
    print(noise, round(stability(dt, X_test, noise), 4))

tree confidence values: [1.]
0.01 0.9932
0.05 0.975
0.1 0.9114
0.2 0.775


In [13]:
shuffled = X_test.apply(lambda col: rng.permutation(col.values))
print("shuffled rows flagged:", (detector.predict(shuffled) == -1).mean().round(3))
print("NB confidence on them:", nb.predict_proba(shuffled).max(axis=1).mean().round(3))

shuffled rows flagged: 0.02
NB confidence on them: 0.944


In [14]:
from sklearn.model_selection import cross_val_predict

proba_cv = cross_val_predict(GaussianNB(), X, y, cv=5, method="predict_proba")
conf_cv = proba_cv.max(axis=1)
correct_cv = nb.classes_[proba_cv.argmax(axis=1)] == y.values
print("wrong predictions:", (~correct_cv).sum())
for t in [0.7, 0.8, 0.9, 0.95]:
    flagged = conf_cv < t
    print(t, flagged[correct_cv].mean().round(3), flagged[~correct_cv].mean().round(3))

wrong predictions: 11
0.7 0.008 0.182
0.8 0.014 0.636
0.9 0.02 0.818
0.95 0.029 0.909


In [15]:
def stability(model, X, y, noise, seed=42):
    rng = np.random.default_rng(seed)
    shaken = X + rng.normal(0, noise * X.std().values, X.shape)
    pred = model.predict(shaken)
    return (pred == model.predict(X)).mean(), (pred == y.values).mean()

for name, m in [("Naive Bayes", nb), ("Decision Tree", dt)]:
    for noise in [0.05, 0.1, 0.2]:
        same, acc = stability(m, X_test, y_test, noise)
        print(name, noise, "same answer:", round(same, 3), "| accuracy:", round(acc, 3))

Naive Bayes 0.05 same answer: 0.993 | accuracy: 0.993
Naive Bayes 0.1 same answer: 0.984 | accuracy: 0.984
Naive Bayes 0.2 same answer: 0.884 | accuracy: 0.884
Decision Tree 0.05 same answer: 0.957 | accuracy: 0.95
Decision Tree 0.1 same answer: 0.911 | accuracy: 0.902
Decision Tree 0.2 same answer: 0.802 | accuracy: 0.802


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

scaler = StandardScaler().fit(X_train)
nn = NearestNeighbors(n_neighbors=6).fit(scaler.transform(X_train))

def novelty(rows):
    d, _ = nn.kneighbors(scaler.transform(rows), n_neighbors=5)
    return d.mean(axis=1)

d_train, _ = nn.kneighbors(scaler.transform(X_train))
limit = np.percentile(d_train[:, 1:].mean(axis=1), 99)    # skip column 0: the row itself

print("normal test rows flagged:", (novelty(X_test) > limit).mean().round(3))
print("shuffled rows flagged:", (novelty(shuffled) > limit).mean().round(3))
print("weird row flagged:", novelty(weird)[0] > limit)

normal test rows flagged: 0.011
shuffled rows flagged: 0.593
weird row flagged: True


# Role B: Findings (Naive Bayes, Decision Tree, Evaluation)

## 1. Accuracy and class-wise performance
- Naive Bayes scored 0.9955 on the 440 test rows, so 438 were correct and only 2 were wrong.
- Both mistakes were the same pair: rice predicted as jute. This lowers rice recall to 0.90 and jute precision to 0.909. Every other crop was perfect.
- When almost every model scores in the high 90s, a single accuracy number cannot separate them. Per-class results and the tests below are more useful.

## 2. Decision Tree depth (underfitting vs overfitting)
| max_depth | Train | Test |
|---|---|---|
| 2 | 0.1364 | 0.1364 |
| 4 | 0.3182 | 0.3182 |
| 6 | 0.583 | 0.5705 |
| 10 | 0.9818 | 0.9636 |
| None | 1.0 | 0.9795 |

- Shallow trees underfit. A tree of depth d has at most 2^d leaves, and 22 crops need at least 22 leaves.
- The unlimited tree reaches train accuracy 1.0 but test accuracy 0.9795. The gap is small, so overfitting is mild on this dataset.

## 3. Is the model's confidence trustworthy?
- Naive Bayes averages 0.993 confidence when right and 0.788 when wrong, so it is less sure when it is about to make a mistake.
- The unlimited Decision Tree reports confidence 1.0 on every row (each leaf holds one crop), so its confidence cannot be used to flag bad predictions.
- Using 5-fold cross-validation over all 2200 rows (11 mistakes), a threshold of 0.9 catches 81.8% of the wrong predictions and flags only 2.0% of correct ones. The sample of mistakes is small, so this is a rough guide.

## 4. Stability under input noise
Noise is Gaussian, scaled to each feature's standard deviation, with the same seed for both models.

| Noise | Naive Bayes (same answer) | Decision Tree (same answer) |
|---|---|---|
| 5% | 0.993 | 0.957 |
| 10% | 0.984 | 0.911 |
| 20% | 0.884 | 0.802 |

Naive Bayes is more stable than the tree at every noise level.

## 5. Flagging unusual inputs
| Detector | Normal test rows flagged | Shuffled (impossible) rows flagged |
|---|---|---|
| IsolationForest | 1.4% | 2.0% |
| Distance to 5 nearest training rows | 1.1% | 59.3% |

- Shuffled rows keep realistic values per feature but scramble the combinations. Naive Bayes was still 94% confident on them, so confidence alone cannot catch them.
- IsolationForest judges features mostly one at a time, so it missed almost all of them. The distance-based detector looks at the whole row and caught about 59%.
- Both detectors flagged the extreme row [500, 500, 500, 60, 100, 14, 1000].

## Limitations
- Only 2 test mistakes (11 in cross-validation), so confidence and threshold estimates are rough.
- The distance detector still misses about 40% of shuffled rows.